In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Optional
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from dataset import Multi30kDataset
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask

from tqdm import tqdm

from nltk.translate.bleu_score import corpus_bleu
from dataset import *
from train import *
import wandb

In [2]:
wandb.init()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/mohamedmafaz/.netrc.
wandb: Currently logged in as: mohdmafaz200303 (mafaz03) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [3]:
# 9kuc8luf

In [2]:
language_dataset = Multi30kDataset(split = "train")
processed_dataset = language_dataset.process_data()

dataset_obj = TranslationDataset(processed_dataset)
dataloader = DataLoader(
    dataset_obj,
    batch_size=5,
    shuffle=True,
    collate_fn=collate_fn
)

Building vocab, please wait...


100%|██████████| 29000/29000 [00:00<00:00, 35777.20it/s]


In [3]:
sample = next(iter(dataloader))

src_sample_txt = ' '.join([language_dataset.de_itos[i.item()] for i in sample[0][0]])
tgt_sample_txt = ' '.join([language_dataset.en_itos[i.item()] for i in sample[1][0]])

print(src_sample_txt)
print(tgt_sample_txt)

<sos> ein mann mit einem großen weißen bart schiebt einen wagen voller müllbeutel und kartons einen bürgersteig hinunter. <eos>
<sos> a man with a large white beard is pushing a cart full of trash bags and cardboard down a sidewalk. <eos>


In [5]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)


# load_checkpoint("checkpoint.pt", transformer)


Building vocab, please wait...


Processing test: 100%|██████████| 1000/1000 [00:00<00:00, 45928.23it/s]


could not load checkpoint: Error(s) in loading state_dict for Transformer:
	Unexpected key(s) in state_dict: "src_embedding.weight", "tgt_embedding.weight", "positional_encodings.pe", "encoder.encoder_layers.0.mha_self_attn.WQ.weight", "encoder.encoder_layers.0.mha_self_attn.WQ.bias", "encoder.encoder_layers.0.mha_self_attn.WK.weight", "encoder.encoder_layers.0.mha_self_attn.WK.bias", "encoder.encoder_layers.0.mha_self_attn.WV.weight", "encoder.encoder_layers.0.mha_self_attn.WV.bias", "encoder.encoder_layers.0.mha_self_attn.WO.weight", "encoder.encoder_layers.0.mha_self_attn.WO.bias", "encoder.encoder_layers.0.layer_norm1.weight", "encoder.encoder_layers.0.layer_norm1.bias", "encoder.encoder_layers.0.layer_norm2.weight", "encoder.encoder_layers.0.layer_norm2.bias", "encoder.encoder_layers.0.linear1.weight", "encoder.encoder_layers.0.linear1.bias", "encoder.encoder_layers.0.linear2.weight", "encoder.encoder_layers.0.linear2.bias", "encoder.encoder_layers.1.mha_self_attn.WQ.weight", "enc

In [8]:
checkpoint = torch.load("checkpoint.pt", map_location="cpu")
transformer.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [9]:
examples = [
    {
        "source": "verschiedene personen überqueren die straße bei einem spirituosengeschäft und einem thailändischen restaurant.",
        "target": "various people crossing the street by a liquor store and thai house."
    },
    {
        "source": "ein mann mit schaufel und besen in der hand sieht zu, wie ein kleiner junge den schuh einer frau putzt.",
        "target": "a man carrying a broom and dustpan is watching while a young boy is shining a woman's shoe."
    },
    {
        "source": "ein kleines kind sitzt auf mehreren kissen und hat ein grünes tablett mit kleingeschnittenem essen vor sich.",
        "target": "a young child sits on some pillows with a green tray of chopped-up food in front of him."
    }
]

# table = wandb.Table(columns=["Source", "Target", "Translated"])

for ex in examples:

    translated = transformer.infer(
        f"<sos> {ex['source']} <eos> <pad> <pad>"
    )

    print(translated)

#     table.add_data(
#         ex["source"],
#         ex["target"],
#         translated
#     )

# wandb.log({
#     "small translations": table
# })

src_sentence:
<sos> verschiedene personen überqueren die straße bei einem spirituosengeschäft und einem thailändischen restaurant. <eos> <pad> <pad>
result:
various people crossing the street by a house.
various people crossing the street by a house.
src_sentence:
<sos> ein mann mit schaufel und besen in der hand sieht zu, wie ein kleiner junge den schuh einer frau putzt. <eos> <pad> <pad>
result:
a little boy is watching as a man and shoe in one of the other one of the broom and a woman's shoe.
a little boy is watching as a man and shoe in one of the other one of the broom and a woman's shoe.
src_sentence:
<sos> ein kleines kind sitzt auf mehreren kissen und hat ein grünes tablett mit kleingeschnittenem essen vor sich. <eos> <pad> <pad>
result:
a young child sits with several green bucket on his head and has a tray in front of him.
a young child sits with several green bucket on his head and has a tray in front of him.


In [8]:
translated

'a man is making some hats out of leaves.'

In [9]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:
sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
# sample_src = src[0].unsqueeze(0)
# sample_tgt = tgt[0].unsqueeze(0)
src_mask = make_src_mask(src).to(device)


src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)


load_checkpoint("checkpoint.pt", transformer)

9

In [ ]:
q_norm, k_norm, v_norm = None, None, None

for name, param in transformer.named_parameters():
    if param.grad is None:
        continue

    grad_norm = param.grad.detach().norm(2).item()

    if "WQ" in name:
        q_norm = grad_norm

    elif "WK" in name:
        k_norm = grad_norm

    elif "WV" in name:
        v_norm = grad_norm

In [ ]:
param.grad

In [ ]:
text = "der vieler"
torch.tensor([language_dataset.de_vocab[i] for i in text.split()]).unsqueeze(0)

torch.Size([1, 2])

In [ ]:
src.shape

torch.Size([5, 12])

In [ ]:
prediction_idx = greedy_decode(transformer, src, src_mask, 40, sos_idx, eos_idx, "cpu", break_at_eos = True)

In [ ]:
no_clean = ' '.join([language_dataset.en_itos[i.item()] for i in prediction_idx[0]])
no_clean.split("<sos>")[-1].strip().split("<eos>")[0].strip()

'three boys are doing different arts. in a field.'

In [ ]:
for j in range(5):
    print("Source: ")
    for i in src[j]:
        print(language_dataset.de_itos[i.item()], end = " ")

    print("\n\nTarget, ground truth: ")
    for i in tgt[j]:
        print(language_dataset.en_itos[i.item()], end = " ")

    print("\n\nTarget, prediction: ")
    for i in prediction_idx[j]:
        print(language_dataset.en_itos[i.item()], end = " ")
    
    print("\n-----\n")

Source: 
<sos> ein junge in einer rot-weißen badehose springt rückwärts in einen schönen pool. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, ground truth: 
<sos> a boy wearing red and white swimming trunks diving backwards in a beautiful pool. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, prediction: 
<sos> a boy in a red and white swim jacket is jumping into a swimming pool. <eos> <eos> pool. <eos> pool. <eos> <eos> <eos> pool. <eos> pool. <eos> 
-----

Source: 
<sos> ein kleiner junge in baseballmontur holt mit einem schläger hinter seinem kopf in richtung eines vor ihm montierten baseballs aus. <eos> <pad> <pad> <pad> <pad> <pad> 

Target, ground truth: 
<sos> a small boy wearing baseball regalia holds a bat behind his head with a baseball mounted in front of him. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, prediction: 
<sos> a

In [ ]:
bleu_score = evaluate_bleu(transformer, dataloader, language_dataset.en_itos)
bleu_score